# 03. RoPE Tutorial | 旋转位置编码教程

**难度：** Medium | **环境：** CPU-first | **标签：** `基础架构`, `位置编码`, `PyTorch` | **目标人群：** 模型微调与工程部署

本节我们将解析大模型当前最主流的位置编码方式：**RoPE (Rotary Position Embedding)**，并亲手用复数形式（Complex Tensor）实现它。这是 LLaMA, Qwen, DeepSeek 的标配！

**关键词：** `RoPE`, `positional encoding`, `complex tensor`

## 前置阅读

**导语：** 如果还没把张量变换、自动求导和注意力直觉理顺，先看下面几页再进入 RoPE 会更顺。

- [05. PyTorch Tensor Fundamentals | PyTorch 张量基础操作](../00_Prerequisites/05_PyTorch_Tensor_Fundamentals.ipynb)
- [07. PyTorch Autograd and Backward | PyTorch 自动求导与反向传播](../00_Prerequisites/07_PyTorch_Autograd_and_Backward.ipynb)
- [16. Attention Mechanism Intro | 注意力机制导论](../00_Prerequisites/16_Attention_Mechanism_Intro.ipynb)

## 相关阅读

**导语：** 本节先把 RoPE 的旋转位置编码数学推导讲清楚；如果想看它和 Attention 融合后在实现层怎么落地，再看硬件直觉和算子融合页面。

- [03. GPU Architecture and Memory | GPU 物理架构与内存层级](../01_Hardware_Math_and_Systems/03_GPU_Architecture_and_Memory.ipynb)
- [19. Operator Fusion Introduction | 算子融合导论](../01_Hardware_Math_and_Systems/19_Operator_Fusion_Introduction.ipynb)


### Step 1: 核心思想与痛点

> **为什么需要 RoPE？**
> 原生的 Transformer 使用绝对位置编码（如正弦波或可学习参数），导致模型很难泛化到比训练集更长的序列。我们希望模型能在计算 Attention 时感知到 Token 之间的**相对距离**。
> **RoPE 的本质：**
> “借用复数的旋转”。通过将 Query 和 Key 的向量映射到复数空间并旋转特定角度，在计算内积（Dot-product）时，结果自然就带有了相对位置信息 $(m-n)$。


### Step 2: 代码实现框架
在 PyTorch 教学代码中，最直观的 RoPE 实现方式之一是利用复数乘法：将最后一维两两配对成复数，再乘以预先计算好的复数旋转矩阵 $e^{im\theta}$，最后用 `torch.view_as_real` 恢复为实数表示。真实高性能实现通常不会依赖 PyTorch complex tensor，而是缓存 `cos/sin` 后直接做 `x * cos + rotate(x) * sin`，因为这种形式更容易和 Q/K projection、KV cache 写入或 attention kernel 融合。


因此实现时的主线其实很固定：先算出 `freqs_cis`，再把它和 `xq / xk` 做广播对齐，最后完成复数旋转并回到原始实数形状。这样学习者在写 `TODO 1/2/3` 时，就能清楚知道每一步是在接哪一段链路。


###  Step 3: 核心公式与张量维度

1. **预计算旋转角 (Precompute Frequencies):**
   频率计算公式：$\Theta = 10000^{-2i/d}$，其中 $i$ 是维度索引，$d$ 是 Head Dimension。
   生成复数形式的极坐标：$e^{i m \Theta} = \cos(m \Theta) + i \sin(m \Theta)$
   
2. **应用旋转 (Apply Rotary Embedding):**
   将输入的 Query 或 Key 视为复数：`x = x_real + i * x_imag`
   利用复数乘法直接完成旋转矩阵的运算：$x_{rotated} = x \times e^{i m \Theta}$

**实现提示：** `reshape_for_broadcast` 的作用是把 `freqs_cis` 调整成和 `xq_ / xk_` 可广播对齐的形状。先对齐维度，再做复数乘法，旋转位置编码才会同时作用到 batch 和 head 维度。


###  Step 4: 动手实战

**要求**：请补全下方 `precompute_freqs_cis` 和 `apply_rotary_emb` 函数。
提示：可以使用 `torch.view_as_complex` 和 `torch.view_as_real` 这两个核心函数！


In [1]:
import torch

In [46]:
def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    """
    计算复数指数频率张量 (cis = cos + i * sin)
    """
    # ==========================================
    # TODO 1: 用极坐标生成复数张量 (提示: torch.polar)
    # ==========================================
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: dim // 2].float() / dim))
    t = torch.arange(end)
    freqs = torch.outer(t, freqs)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    # shape (end, dim // 2)
    return freqs_cis

# 将频率张量扩展到可广播形状，供 Step 3 的复数乘法使用
def reshape_for_broadcast(freqs_cis: torch.Tensor, x: torch.Tensor):
    ndim = x.ndim
    # x shape: b,s,h,d
    # f shape: 1,s,1,d
    shape = [d if i == 1 or i == ndim - 1 else 1 for i, d in enumerate(x.shape)]
    print("input x shape:", x.shape)
    print("input cis shape:", freqs_cis.shape)
    print("target shape: ", shape)
    return freqs_cis.view(*shape)

def apply_rotary_emb(
    xq: torch.Tensor,
    xk: torch.Tensor,
    freqs_cis: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    将旋转位置编码应用到 Query 和 Key 上
    """
    # ==========================================
    # TODO 2: 将 xq, xk 从实数张量转为复数张量
    # 提示: 先把最后一维拆成两个一组，再转成复数
    # xq_ = ???
    # xk_ = ???
    # ==========================================
    # TODO 2: 转换为复数张量（注意精度提升）
    print("input xq:", xq.shape)
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    print("output xq_:", xq_.shape, xq_[0,0,0,:])
    # b,s,nh,nd/2
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))

    # b,s,nh,nd
    freqs_cis = reshape_for_broadcast(freqs_cis, xq_)
    
    # TODO 3: 复数乘法并转回实数
    # shape $[B, S, H, \frac{D}{2}, 2]$ -> flatten start from 3
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    print("xq out:", xq_out[0,0,0])
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(start_dim=3)
    
    return xq_out.type_as(xq), xk_out.type_as(xk)


In [47]:
precompute_freqs_cis(10, 5).shape

torch.Size([5, 5])

In [48]:
precompute_freqs_cis(10, 5)[2]

tensor([-0.4161+0.9093j,  0.9502+0.3117j,  0.9987+0.0502j,  1.0000+0.0080j,
         1.0000+0.0013j])

In [49]:
# 运行此单元格以测试你的实现
def test_rope():
    try:
        print("=" * 60)
        print("开始测试 RoPE 旋转位置编码")
        print("=" * 60)

        batch_size, seq_len, num_heads, head_dim = 2, 16, 4, 64

        # Test 1: 形状测试
        print("\n【Test 1】形状测试")
        xq = torch.randn(batch_size, seq_len, num_heads, head_dim)
        xk = torch.randn(batch_size, seq_len, num_heads, head_dim)

        freqs_cis = precompute_freqs_cis(head_dim, seq_len)
        xq_out, xk_out = apply_rotary_emb(xq, xk, freqs_cis)

        assert xq_out.shape == xq.shape, f"Query 输出形状错误: 期望 {xq.shape}, 实际 {xq_out.shape}"
        assert xk_out.shape == xk.shape, f"Key 输出形状错误: 期望 {xk.shape}, 实际 {xk_out.shape}"
        assert freqs_cis.shape == (seq_len, head_dim // 2), f"频率张量形状错误"
        
        # 核心修复：防止占位符作弊，输出绝不能等于输入
        assert not torch.allclose(xq, xq_out, atol=1e-5), "TODO 3 未完成: 输出与输入完全相同，RoPE 旋转未生效！"
        
        print("  ✅ 输出形状测试通过")
        print("  ✅ 频率张量形状测试通过")

        # Test 2: 数值范围测试
        print("\n【Test 2】数值范围测试")
        norm_before = torch.norm(xq, dim=-1)
        norm_after = torch.norm(xq_out, dim=-1)
        assert torch.allclose(norm_before, norm_after, rtol=1e-4, atol=1e-5), "RoPE 改变了向量模长！"
        print("  ✅ 向量模长保持不变（旋转不变性）")

        assert not torch.isnan(xq_out).any(), "输出包含 NaN！"
        assert not torch.isinf(xq_out).any(), "输出包含 Inf！"
        print("  ✅ 无 NaN/Inf 数值异常")

        # Test 3: 相对位置编码验证
        print("\n【Test 3】相对位置编码验证")
        pos0 = xq_out[:, 0, :, :]
        pos1 = xq_out[:, 1, :, :]
        assert not torch.allclose(pos0, pos1, rtol=1e-3), "不同位置的输出相同，位置编码失败！"
        print("  ✅ 位置编码生效（不同位置输出不同）")

        # Test 4: 精度稳定性测试
        print("\n【Test 4】精度稳定性测试")
        xq_fp16 = torch.randn(1, 8, 2, head_dim, dtype=torch.float16)
        xk_fp16 = torch.randn(1, 8, 2, head_dim, dtype=torch.float16)
        freqs_fp16 = precompute_freqs_cis(head_dim, 8)

        xq_out_fp16, xk_out_fp16 = apply_rotary_emb(xq_fp16, xk_fp16, freqs_fp16)

        assert xq_out_fp16.dtype == torch.float16, "输出类型错误！"
        assert not torch.isnan(xq_out_fp16).any(), "FP16 输入导致 NaN！"
        print("  ✅ FP16 输入处理正确")
        print("  ✅ 精度提升机制工作正常")

        print("\n" + "=" * 60)
        print(" RoPE 算子实现通过测试。")
        print("   所有测试用例均已通过")
        print("=" * 60)

    except NotImplementedError:
        print("\n❌ 测试失败: 请先完成 TODO 部分的代码！")
        raise
    except (AttributeError, NameError, TypeError) as e:
        print(f"\n❌ 测试失败: 代码可能未完成")
        raise NotImplementedError("请先完成 TODO 部分的代码！") from e
    except AssertionError as e:
        print(f"\n❌ 测试失败: {e}")
        raise
    except Exception as e:
        print(f"\n❌ 发生未知异常: {type(e).__name__}: {e}")
        raise

test_rope()


开始测试 RoPE 旋转位置编码

【Test 1】形状测试
input xq: torch.Size([2, 16, 4, 64])
output xq_: torch.Size([2, 16, 4, 32]) tensor([ 2.1533+0.7877j, -0.2251+0.5342j,  2.8161-0.5383j, -1.7693+0.4426j,
         1.4940-1.6430j,  0.5708-0.7307j,  0.1052+0.0694j, -0.6569-0.9169j,
         0.2247-2.4507j,  0.2264+0.5445j,  0.0944+0.6046j, -1.1426+0.6824j,
         2.9102+0.6398j, -0.8297+1.3361j,  0.8381+0.2080j, -0.2002-0.1226j,
         1.8465+0.4688j, -0.2481-0.7347j,  1.1461+1.4462j,  1.7421-0.4492j,
         0.6625+0.6439j, -0.6012-1.0587j, -1.1092+0.7581j, -0.0595+1.3119j,
        -2.0475+0.5346j, -1.0607+1.1139j,  0.8766+0.2562j, -0.9362-0.8240j,
         1.1089-0.0861j,  0.8261+1.1068j, -1.2156+0.2056j, -0.6681+1.0439j])
input x shape: torch.Size([2, 16, 4, 32])
input cis shape: torch.Size([16, 32])
target shape:  [1, 16, 1, 32]
xq out: tensor([ 2.1533,  0.7877, -0.2251,  0.5342,  2.8161, -0.5383, -1.7693,  0.4426,
         1.4940, -1.6430,  0.5708, -0.7307,  0.1052,  0.0694, -0.6569, -0.9169,
      

---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果你正在 Colab 中运行，并且遇到困难没有思路，可以向下滚动查看参考答案。
<br><br><br><br><br><br><br><br><br><br>

---

## 参考代码与解析

### 代码

In [ ]:
def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    # TODO 1: 计算逆频率并生成复数张量
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_cis

def reshape_for_broadcast(freqs_cis: torch.Tensor, x: torch.Tensor):
    ndim = x.ndim
    shape = [d if i == 1 or i == ndim - 1 else 1 for i, d in enumerate(x.shape)]
    return freqs_cis.view(*shape)

def apply_rotary_emb(
    xq: torch.Tensor,
    xk: torch.Tensor,
    freqs_cis: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    # TODO 2: 转换为复数张量（注意精度提升）
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    
    freqs_cis = reshape_for_broadcast(freqs_cis, xq_)
    
    # TODO 3: 复数乘法并转回实数
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    
    return xq_out.type_as(xq), xk_out.type_as(xk)

### 解析

**1. TODO 1 (预计算旋转频率与极坐标复数生成)**

- **逆频率计算：** 使用公式 $\Theta = 10000^{-2i/d}$ 计算每个维度的旋转频率。代码中  步长为 2，对应复数的实部和虚部配对，除以  后取负指数。
- **位置编码矩阵：** 通过  生成  的角度矩阵，其中  是位置索引 。
- **极坐标复数：**  生成复数 ^{i\theta}$，这里  全为 1（模长）， 是预计算的角度矩阵。这是 RoPE 的核心数学表示。
- **工程细节：** 为什么代码用  而公式是 hmtBc2i/d$？因为 PyTorch 复数将最后一维按  成对存储，步长 2 正好对应公式中的  指数。

**2. TODO 2 (实数张量转复数张量与精度提升)**

- **精度提升的必要性（Critical）：** 在执行  之前必须先调用  将张量提升到 FP32。这是因为复数乘法在 FP16/BF16 下极易发散或产生 NaN，导致训练崩溃。这是 RoPE 实现中最容易踩的坑，LLaMA 等开源模型的源码中都强制使用 FP32 进行旋转计算。
- **维度重塑：**  将最后一维  拆分为 ，其中 2 对应实部和虚部。
- **复数转换：**  将形状  的实数张量解释为复数张量 ，每两个相邻元素组成一个复数。

**3. TODO 3 (复数乘法旋转与实数还原)**

- **广播机制：**  将  的形状从  扩展为 ，以便与  的形状  进行广播。
- **复数乘法：**  完成旋转操作，这是 RoPE 的核心计算。复数乘法 (c+di) = (ac-bd) + (ad+bc)i$ 自动实现了旋转矩阵的效果。
- **实数还原：**  将复数张量转回实数表示，在最后增加一个大小为 2 的维度 。
- **维度展平：**  将最后两个维度  合并回 ，恢复原始形状。
- **类型恢复：**  将结果转回输入的原始精度（如 FP16），因为前面为了数值稳定性提升到了 FP32。

**进阶思考：RoPE 的上下文外推 (Context Extension)**

- **问题背景：** 模型在 4K 序列长度上训练，如何在推理时支持 16K 甚至 128K？直接外推会导致性能急剧下降。
- **解决方案：** 工业界提出了多种 RoPE Scaling 技术：
  - **线性插值 (Linear Scaling)：** 将位置索引  除以缩放因子，相当于压缩位置空间。
  - **NTK-aware Scaling：** 动态调整基频 （如从 10000 增大到 100000），降低高频分量的旋转速度。
  - **YaRN：** 结合低频外推和高频插值，在不同维度使用不同的缩放策略。
- **工程实践：** LLaMA 2 使用线性插值支持 32K 上下文，Qwen 使用动态 NTK 支持 128K，这些技术使得 RoPE 成为当前大模型位置编码的事实标准。

---

## 额外练习

下面两个额外练习延续前面的 notebook 风格：先给你练习版，把关键部分 mask 成 `TODO`；你可以自己补完并运行测试。参考答案统一放在后面的“额外练习参考答案”部分。

练习顺序按真实工程常见度来排：

1. **生产常用的 `cos/sin` RoPE 写法**：理解为什么真实 kernel 更常用 real tensor 乘加，而不是 PyTorch complex tensor。
2. **Dynamic NTK RoPE Scaling + Max Context + Cache**：理解超过 `max_context` 时如何扩展 RoPE cache。


### 额外练习 1：生产常用的 `cos/sin` RoPE 写法

复数形式的好处是数学含义非常清楚：

$$
(a + ib)(\cos\theta + i\sin\theta)
= (a\cos\theta - b\sin\theta) + i(a\sin\theta + b\cos\theta)
$$

如果把相邻两个维度 `(a, b)` 看成一个复数，那么直接实现时只需要两步：

1. 把 `(a, b)` 旋转成 `(-b, a)`。
2. 计算 `x * cos + rotate(x) * sin`。

请补全下面的 TODO，并让测试通过。


In [ ]:
# 额外练习 1: 用 cos/sin 实现 RoPE

def rotate_adjacent_pairs(x):
    """For adjacent-pair RoPE layout: [a, b] -> [-b, a]."""
    x_pair = x.reshape(*x.shape[:-1], -1, 2)

    # TODO 1: 将每一对 [a, b] 变成 [-b, a]
    x_rotated = ...

    return x_rotated.flatten(start_dim=-2)


def apply_rope_with_cos_sin(xq, xk, freqs_cis):
    """Apply RoPE with cached cos/sin instead of complex tensors."""
    # TODO 2: 从 freqs_cis 取出 cos/sin，并把 [S, D/2] 扩成 [S, D]
    cos = ...
    sin = ...

    # TODO 3: reshape 成可以 broadcast 到 [B, S, H, D] 的形状
    cos = ...
    sin = ...

    # TODO 4: 使用 x * cos + rotate(x) * sin 完成 Q/K 旋转
    xq_out = ...
    xk_out = ...

    return xq_out.type_as(xq), xk_out.type_as(xk)



In [ ]:
# 运行此单元格测试额外练习 1

def apply_rope_with_complex_for_test(xq, xk, freqs_cis):
    xq_complex = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_complex = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = freqs_cis.view(1, freqs_cis.shape[0], 1, freqs_cis.shape[1])
    xq_out = torch.view_as_real(xq_complex * freqs_cis).flatten(start_dim=3)
    xk_out = torch.view_as_real(xk_complex * freqs_cis).flatten(start_dim=3)
    return xq_out.type_as(xq), xk_out.type_as(xk)


def test_cos_sin_rope():
    batch_size, seq_len, num_heads, head_dim = 2, 16, 4, 64
    xq = torch.randn(batch_size, seq_len, num_heads, head_dim)
    xk = torch.randn(batch_size, seq_len, num_heads, head_dim)
    freqs_cis = precompute_freqs_cis(head_dim, seq_len)

    xq_complex, xk_complex = apply_rope_with_complex_for_test(xq, xk, freqs_cis)
    xq_direct, xk_direct = apply_rope_with_cos_sin(xq, xk, freqs_cis)

    assert torch.allclose(xq_complex, xq_direct, rtol=1e-5, atol=1e-6)
    assert torch.allclose(xk_complex, xk_direct, rtol=1e-5, atol=1e-6)
    assert torch.allclose(torch.norm(xq, dim=-1), torch.norm(xq_direct, dim=-1), rtol=1e-4, atol=1e-5)
    print("cos/sin RoPE test passed.")


test_cos_sin_rope()



### 额外练习 2：Dynamic NTK RoPE Scaling + Max Context + Cache

真实模型里通常不会每次 forward 都重新算 RoPE。更常见的做法是：

1. 按模型原始训练长度 `max_context` 预计算一份 RoPE cache。
2. 如果当前 `position_ids` 没有超过 `max_context`，直接从 cache 里取对应行。
3. 如果推理长度超过 `max_context`，使用 RoPE scaling 规则扩展 cache，再按 `position_ids` 取对应行。

这里练一个常见、代码也比较短的方案：**Dynamic NTK-aware scaling**。

核心直觉：位置 `m` 变大以后，高频维度会转得太快。Dynamic NTK 的做法不是简单地把位置取模，也不是直接截断，而是在超过原始上下文时动态增大 RoPE 的 base，也就是代码里的 `theta`。base 变大后，很多维度的 `inv_freq` 会变小，旋转速度会放慢。


In [ ]:
# 额外练习 2: Dynamic NTK RoPE cache

def build_inv_freq(dim, theta=10000.0, device=None):
    return 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))


def build_freqs_cis_from_inv_freq(inv_freq, end):
    t = torch.arange(end, device=inv_freq.device, dtype=torch.float32)
    freqs = torch.outer(t, inv_freq.float())
    return torch.polar(torch.ones_like(freqs), freqs)


def dynamic_ntk_inv_freq(
    dim,
    seq_len,
    max_context,
    theta=10000.0,
    scaling_factor=4.0,
    device=None,
):
    """Return scaled inv_freq and scaled theta/base."""
    if dim <= 2:
        raise ValueError("dynamic NTK formula needs dim > 2")

    # TODO 1: seq_len 没超过 max_context 时，effective_seq_len 应该至少等于 max_context
    effective_seq_len = ...

    # TODO 2: 根据 Dynamic NTK 公式增大 theta/base
    scaled_theta = ...

    # TODO 3: 用 scaled_theta 重新计算 inv_freq
    inv_freq = ...

    return inv_freq, scaled_theta


class DynamicNTKRoPECache:
    """Small teaching version of a production RoPE cache."""

    def __init__(self, dim, max_context, theta=10000.0, scaling_factor=4.0, device=None):
        self.dim = dim
        self.max_context = max_context
        self.theta = theta
        self.scaling_factor = scaling_factor
        self.max_seq_len_cached = max_context
        self.scaled_theta = theta

        inv_freq = build_inv_freq(dim=dim, theta=theta, device=device)
        self.freqs_cis = build_freqs_cis_from_inv_freq(inv_freq, max_context)

    def get(self, position_ids):
        """
        Return RoPE rows for absolute positions.

        prefill: position_ids = torch.arange(seq_len)
        decode:  position_ids = torch.tensor([absolute_position])
        """
        position_ids = position_ids.to(device=self.freqs_cis.device, dtype=torch.long)

        # TODO 4: 根据 position_ids 算出至少需要多长的 cache
        required_len = ...

        # TODO 5: 如果 required_len 超过当前 cache，就用 Dynamic NTK 重建更长 cache
        if ...:
            inv_freq, self.scaled_theta = ...
            self.freqs_cis = ...
            self.max_seq_len_cached = ...

        # TODO 6: 返回 position_ids 对应的 RoPE rows，而不是简单返回 [:seq_len]
        return ...



In [ ]:
# 运行此单元格测试额外练习 2

def test_dynamic_ntk_rope_cache():
    head_dim = 64
    max_context = 16
    rope_cache = DynamicNTKRoPECache(
        dim=head_dim,
        max_context=max_context,
        theta=10000.0,
        scaling_factor=4.0,
    )

    short_freqs = rope_cache.get(torch.arange(0, 8))
    assert short_freqs.shape == (8, head_dim // 2)
    assert rope_cache.max_seq_len_cached == max_context
    assert rope_cache.scaled_theta == 10000.0

    long_freqs = rope_cache.get(torch.arange(0, 40))
    assert long_freqs.shape == (40, head_dim // 2)
    assert rope_cache.max_seq_len_cached == 40
    assert rope_cache.scaled_theta > 10000.0

    decode_freq = rope_cache.get(torch.tensor([39]))
    assert torch.allclose(decode_freq, rope_cache.freqs_cis[39:40])
    assert not torch.allclose(decode_freq, rope_cache.freqs_cis[:1])

    print("dynamic NTK RoPE cache test passed.")


test_dynamic_ntk_rope_cache()



---

🛑 **STOP HERE** 🛑

请先补全上面的两个额外练习并跑通测试。下面是参考答案。


## 额外练习参考答案

### 答案 1：生产常用的 `cos/sin` RoPE 写法


In [ ]:
def rotate_adjacent_pairs(x):
    """For adjacent-pair RoPE layout: [a, b] -> [-b, a]."""
    x_pair = x.reshape(*x.shape[:-1], -1, 2)
    x_rotated = torch.stack((-x_pair[..., 1], x_pair[..., 0]), dim=-1)
    return x_rotated.flatten(start_dim=-2)


def apply_rope_with_cos_sin(xq, xk, freqs_cis):
    """Apply RoPE with cached cos/sin instead of complex tensors."""
    cos = freqs_cis.real.repeat_interleave(2, dim=-1)
    sin = freqs_cis.imag.repeat_interleave(2, dim=-1)
    cos = cos.view(1, cos.shape[0], 1, cos.shape[1])
    sin = sin.view(1, sin.shape[0], 1, sin.shape[1])

    xq_out = xq.float() * cos + rotate_adjacent_pairs(xq.float()) * sin
    xk_out = xk.float() * cos + rotate_adjacent_pairs(xk.float()) * sin
    return xq_out.type_as(xq), xk_out.type_as(xk)


def apply_rope_with_complex_for_test(xq, xk, freqs_cis):
    xq_complex = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_complex = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = freqs_cis.view(1, freqs_cis.shape[0], 1, freqs_cis.shape[1])
    xq_out = torch.view_as_real(xq_complex * freqs_cis).flatten(start_dim=3)
    xk_out = torch.view_as_real(xk_complex * freqs_cis).flatten(start_dim=3)
    return xq_out.type_as(xq), xk_out.type_as(xk)


# Reference check: direct cos/sin should match the complex formulation.
batch_size, seq_len, num_heads, head_dim = 2, 16, 4, 64
xq = torch.randn(batch_size, seq_len, num_heads, head_dim)
xk = torch.randn(batch_size, seq_len, num_heads, head_dim)
freqs_cis = precompute_freqs_cis(head_dim, seq_len)

xq_complex, xk_complex = apply_rope_with_complex_for_test(xq, xk, freqs_cis)
xq_direct, xk_direct = apply_rope_with_cos_sin(xq, xk, freqs_cis)

print("max q diff:", (xq_complex - xq_direct).abs().max().item())
print("max k diff:", (xk_complex - xk_direct).abs().max().item())

assert torch.allclose(xq_complex, xq_direct, rtol=1e-5, atol=1e-6)
assert torch.allclose(xk_complex, xk_direct, rtol=1e-5, atol=1e-6)




### 答案 2：Dynamic NTK RoPE Scaling + Max Context + Cache


In [ ]:
def build_inv_freq(dim, theta=10000.0, device=None):
    return 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))


def build_freqs_cis_from_inv_freq(inv_freq, end):
    t = torch.arange(end, device=inv_freq.device, dtype=torch.float32)
    freqs = torch.outer(t, inv_freq.float())
    return torch.polar(torch.ones_like(freqs), freqs)


def dynamic_ntk_inv_freq(
    dim,
    seq_len,
    max_context,
    theta=10000.0,
    scaling_factor=4.0,
    device=None,
):
    if dim <= 2:
        raise ValueError("dynamic NTK formula needs dim > 2")

    effective_seq_len = max(seq_len, max_context)
    scaled_theta = theta * (
        (scaling_factor * effective_seq_len / max_context) - (scaling_factor - 1)
    ) ** (dim / (dim - 2))
    inv_freq = build_inv_freq(dim=dim, theta=scaled_theta, device=device)
    return inv_freq, scaled_theta


class DynamicNTKRoPECache:
    """Small teaching version of a production RoPE cache."""

    def __init__(self, dim, max_context, theta=10000.0, scaling_factor=4.0, device=None):
        self.dim = dim
        self.max_context = max_context
        self.theta = theta
        self.scaling_factor = scaling_factor
        self.max_seq_len_cached = max_context
        self.scaled_theta = theta

        inv_freq = build_inv_freq(dim=dim, theta=theta, device=device)
        self.freqs_cis = build_freqs_cis_from_inv_freq(inv_freq, max_context)

    def get(self, position_ids):
        position_ids = position_ids.to(device=self.freqs_cis.device, dtype=torch.long)
        required_len = int(position_ids.max().item()) + 1

        if required_len > self.max_seq_len_cached:
            inv_freq, self.scaled_theta = dynamic_ntk_inv_freq(
                dim=self.dim,
                seq_len=required_len,
                max_context=self.max_context,
                theta=self.theta,
                scaling_factor=self.scaling_factor,
                device=self.freqs_cis.device,
            )
            self.freqs_cis = build_freqs_cis_from_inv_freq(inv_freq, required_len)
            self.max_seq_len_cached = required_len

        return self.freqs_cis[position_ids]


# Reference check: short requests reuse cache; long requests grow cache with Dynamic NTK.
head_dim = 64
max_context = 16
rope_cache = DynamicNTKRoPECache(head_dim, max_context, theta=10000.0, scaling_factor=4.0)

short_freqs = rope_cache.get(torch.arange(0, 8))
print("short freqs shape:", short_freqs.shape)
print("cached length after short request:", rope_cache.max_seq_len_cached)
print("theta/base after short request:", rope_cache.scaled_theta)

long_freqs = rope_cache.get(torch.arange(0, 40))
print("long freqs shape:", long_freqs.shape)
print("cached length after long request:", rope_cache.max_seq_len_cached)
print("theta/base after long request:", rope_cache.scaled_theta)

assert short_freqs.shape == (8, head_dim // 2)
assert long_freqs.shape == (40, head_dim // 2)
assert rope_cache.max_seq_len_cached == 40
assert rope_cache.scaled_theta > 10000.0

# Decode should use the absolute row, not row 0.
decode_freq = rope_cache.get(torch.tensor([39]))
assert torch.allclose(decode_freq, rope_cache.freqs_cis[39:40])
assert not torch.allclose(decode_freq, rope_cache.freqs_cis[:1])



### 你应该记住什么

- 复数写法适合理解 RoPE 的数学本质；生产高性能路径更常用 `cos/sin + rotate`。
- RoPE cache 存的是每个绝对位置的旋转因子，形状通常类似 `[max_seq_len_cached, head_dim // 2]`。
- prefill 阶段可以取 `position_ids = 0..seq_len-1`；decode 阶段必须取新 token 的**绝对位置**。
- Dynamic NTK scaling 在 `seq_len > max_context` 时动态增大 `theta/base`，让旋转频率变慢，再重建更长的 cache。
- 超过 `max_context` 时不能简单地 `position % max_context`，那会破坏位置语义。
